In [30]:
import statistics
import pandas as pd
import sys
sys.path.append("..")
import equity as eq
import equity_strats as eq_strats
import matplotlib.pyplot as plt
import math

In [23]:
XLK_options_csv = '../data/XLK_options.csv'
XLK_underlying_csv = '../data/XLK_underlying.csv'
XLK_options = pd.read_csv(XLK_options_csv, parse_dates = ["exdate", "date"])
XLK_underlying = pd.read_csv(XLK_underlying_csv, parse_dates = ["date"])
SPY_options_csv = '../data/SPY_options.csv'
SPY_underlying_csv = '../data/SPY_underlying.csv'
SPY_options = pd.read_csv(SPY_options_csv, parse_dates = ["exdate", "date"])
SPY_underlying = pd.read_csv(SPY_underlying_csv, parse_dates = ["date"])
AAPL_options_csv = '../data/AAPL_options.csv'
AAPL_underlying_csv = '../data/AAPL_underlying.csv'
AAPL_options = pd.read_csv(AAPL_options_csv, parse_dates = ["exdate", "date"])
AAPL_underlying = pd.read_csv(AAPL_underlying_csv, parse_dates = ["date"])
AVGO_options_csv = '../data/avgo_options.csv'
AVGO_underlying_csv = '../data/avgo_underlying.csv'
AVGO_options = pd.read_csv(AVGO_options_csv, parse_dates = ["exdate", "date"])
AVGO_underlying = pd.read_csv(AVGO_underlying_csv, parse_dates = ["date"])
NVDA_options_csv = '../data/nvda_options.csv'
NVDA_underlying_csv = '../data/nvda_underlying.csv'
NVDA_options = pd.read_csv(NVDA_options_csv, parse_dates = ["exdate", "date"])
NVDA_underlying = pd.read_csv(NVDA_underlying_csv, parse_dates = ["date"])
MSFT_options_csv = '../data/MSFT_options.csv'
MSFT_underlying_csv = '../data/MSFT_underlying.csv'
MSFT_options = pd.read_csv(MSFT_options_csv, parse_dates = ["exdate", "date"])
MSFT_underlying = pd.read_csv(MSFT_underlying_csv, parse_dates = ["date"])
CRM_options_csv = '../data/CRM_options.csv'
CRM_underlying_csv = '../data/CRM_underlying.csv'
CRM_options = pd.read_csv(CRM_options_csv, parse_dates = ["exdate", "date"])
CRM_underlying = pd.read_csv(CRM_underlying_csv, parse_dates = ["date"])
VZ_options_csv = '../data/VZ_options.csv'
VZ_underlying_csv = '../data/VZ_underlying.csv'
VZ_options = pd.read_csv(VZ_options_csv, parse_dates = ["exdate", "date"])
VZ_underlying = pd.read_csv(VZ_underlying_csv, parse_dates = ["date"])
risk_free = pd.read_csv("../data/DGS10.csv", parse_dates = ["DATE"])

In [28]:
XLK_underlying_weightings = {
    "MSFT": 0.2339,
    "AAPL": 0.1996,
    "AVGO": 0.0456,
    "NVDA": 0.0448,
    "CRM": 0.0295
}

XLK_underlying_dfs = {
    "MSFT": MSFT_underlying,
    "AAPL": AAPL_underlying,
    "AVGO": AVGO_underlying,
    "NVDA": NVDA_underlying,
    "CRM": CRM_underlying
}

XLK_options_dfs = {
    "MSFT": MSFT_options,
    "AAPL": AAPL_options,
    "AVGO": AVGO_options,
    "NVDA": NVDA_options,
    "CRM": CRM_options
}




In [37]:
etf_options_df: pd.DataFrame = XLK_options
etf_underlying_df: pd.DataFrame = XLK_underlying
stock_options_dataframes: dict[str, pd.DataFrame] = XLK_options_dfs
stock_underlying_dataframes: dict[str, pd.DataFrame] = XLK_underlying_dfs
etf_weightings: dict[str, float] = XLK_underlying_weightings
date = pd.Timestamp(2018, 1, 2)
target_expiry = 180
tol = 0.3
num_options = 10


# Calculate ATM options for ETF
etf_atm_options = eq.find_atm(etf_options_df, etf_underlying_df, date, 1, target_expiry, tol)

# Current etf_weightings don't sum to 1, so we scale them
print("HERE")
scaled_etf_weightings = {stock: etf_weightings[stock] / sum(etf_weightings.values()) for stock in etf_weightings}

# Calculate total ETF premium from ATM options
total_etf_premium = 0
for call_id, put_id in etf_atm_options:
    call_data = etf_options_df.loc[etf_options_df['symbol'] == call_id].iloc[0]
    put_data = etf_options_df.loc[etf_options_df['symbol'] == put_id].iloc[0]
    call_premium = ((call_data['best_bid'] + call_data['best_offer']) / 2) * call_data['contract_size']
    put_premium = ((put_data['best_bid'] + put_data['best_offer']) / 2) * put_data['contract_size']
    total_etf_premium += call_premium + put_premium # Plus because selling options on index

total_etf_premium *= num_options  # Adjust for the number of options we buy/sell
print(total_etf_premium)

# Calculate total premium for each stock
stock_premiums = {}
total_weighted_premiums = 0
for stock, stock_options_df in stock_options_dataframes.items():
    stock_underlying_df = stock_underlying_dataframes[stock]
    stock_atm_options = eq.find_atm(stock_options_df, stock_underlying_df, date, 1, target_expiry, tol)
    
    total_stock_premium = 0
    for call_id, put_id in stock_atm_options:
        call_data = stock_options_df.loc[stock_options_df['symbol'] == call_id].iloc[0]
        put_data = stock_options_df.loc[stock_options_df['symbol'] == put_id].iloc[0]
        call_premium = ((call_data['best_bid'] + call_data['best_offer']) / 2) * call_data['contract_size']
        put_premium = ((put_data['best_bid'] + put_data['best_offer']) / 2) * put_data['contract_size']
        total_stock_premium += call_premium + put_premium
    
    stock_premiums[stock] = total_stock_premium
    total_weighted_premiums += total_stock_premium * scaled_etf_weightings[stock]

print(stock_premiums)

c = total_etf_premium / total_weighted_premiums    

# Get the number of straddles for each stock by C * scaled_weighting.
allocation_results = {stock: int(round(c * scaled_etf_weightings[stock] * num_options)) for stock in stock_premiums}
print(allocation_results)


HERE
10135.0
{'MSFT': 1210.5000000000002, 'AAPL': 2877.5, 'AVGO': 4240.0, 'NVDA': 4215.0, 'CRM': 2639.5}
{'MSFT': 18, 'AAPL': 15, 'AVGO': 4, 'NVDA': 3, 'CRM': 2}


In [39]:
portfolio: dict[str, float] = {}

portfolio = eq.calculate_allocation_premium_neutral(date, etf_options_df, etf_underlying_df, stock_options_dataframes, stock_underlying_dataframes, allocation_results, num_options, target_expiry, tol)

TypeError: argument of type 'method' is not iterable